# Transaction Anomaly Detection — EDA & Data Quality

1,000,000 transactions, 2025, one merchant, no `is_anomaly` label — unsupervised anomaly detection.

**Notebook map:**
1. `01_eda_data_quality.ipynb` — this notebook: structure, integrity, baseline
2. `02_anomaly_detection.ipynb` — the 4 confirmed anomaly clusters
3. `03_ml_scoring.ipynb` — IsolationForest cross-check
4. `04_conclusions_submission.ipynb` — summary + `submission.csv`

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False,
                      'axes.spines.right': False, 'figure.facecolor': 'white'})

DATA_PATH = '/home/veronika/Anomaly_Hunter_Solidgate/hackathon_int20h_dataset_test.csv'

df = pd.read_csv(DATA_PATH)
df['created_at'] = pd.to_datetime(df['created_at'])
df['processed_at'] = pd.to_datetime(df['processed_at'])

In [2]:
print(f"{df.shape[0]:,} rows x {df.shape[1]} columns")
df.head(3)

1,000,000 rows x 18 columns


,created_at,order_id,processed_at,order_type,user_id,ip_country,currency,amount,payment_method,order_payment_type,bin_country,bank_id,psp_id,has_refund,refunded_amount,is_secured,status,error_code
0,2025-07-01 09:21:23,1,2025-07-01 09:21:32,first,692925,DEU,EUR,4.60,googlepay,NaN,GBR,32,psp_alpha,False,0.0,False,fail,3.02
1,2025-09-01 01:15:47,2,2025-09-01 01:15:57,recurring,452913,CAN,CAD,54.80,card,recurring,CAN,1,psp_alpha,False,0.0,False,success,NaN
2,2025-06-24 23:38:35,3,2025-06-24 23:38:39,first,784680,USA,USD,9.99,card,NaN,USA,32,psp_alpha,False,0.0,False,fail,2.01


In [3]:
missing = df.isna().sum()
missing = missing[missing > 0]
print("Duplicate order_id:", df['order_id'].duplicated().sum())
print("\nMissing values:")
print(missing.to_string())

Duplicate order_id: 0

Missing values:
order_payment_type    400526
error_code            525114


**Structure**
- clean primary key: 0 duplicate `order_id`
- `order_payment_type` missing for 400,526 rows, `error_code` missing for 525,114 — checked below, not data quality issues

In [4]:
rec_check = (df['order_type'] == 'recurring').eq(df['order_payment_type'].notna()).all()
err_check = (df['status'] == 'fail').eq(df['error_code'].notna()).all()
print("order_payment_type set <=> order_type == 'recurring' :", rec_check)
print("error_code set        <=> status == 'fail'            :", err_check)

order_payment_type set <=> order_type == 'recurring' : True
error_code set        <=> status == 'fail'            : True


- confirmed exact (not approximate): both missing-value blocks are structural, not defects

In [5]:
violations = {
    'processed_at before created_at': (df['processed_at'] < df['created_at']).sum(),
    "has_refund=False but refunded_amount>0": ((~df['has_refund']) & (df['refunded_amount'] > 0)).sum(),
    "has_refund=True but refunded_amount==0": (df['has_refund'] & (df['refunded_amount'] == 0)).sum(),
    'zero-amount transactions': (df['amount'] == 0).sum(),
    'refunded_amount > amount': (df['refunded_amount'] > df['amount']).sum(),
}
print("Hard-invariant check (exact violation counts):")
for rule, n in violations.items():
    flag = "OK" if n == 0 else f"-> {n} rows, investigated in notebook 02"
    print(f"  {rule:<42} {n:>8,}   {flag}")

Hard-invariant check (exact violation counts):
  processed_at before created_at                    0   OK
  has_refund=False but refunded_amount>0            0   OK
  has_refund=True but refunded_amount==0            0   OK
  zero-amount transactions                          0   OK
  refunded_amount > amount                      2,691   -> 2691 rows, investigated in notebook 02


**Integrity summary**
- 4 of 5 invariants hold with zero exceptions across all 1,000,000 rows
- the 5th (`refunded_amount > amount`, 2,691 rows) is a real anomaly cluster → `02_anomaly_detection.ipynb`

In [6]:
print(df.nunique().to_string())

created_at             983310
order_id              1000000
processed_at           983301
order_type                  2
user_id                603337
ip_country                  8
currency                    7
amount                    112
payment_method              3
order_payment_type          4
bin_country                 8
bank_id                    50
psp_id                      3
has_refund                  2
refunded_amount           201
is_secured                  2
status                      2
error_code                 13


**Baseline scan** — `bank_id` has 50 distinct values; 49 are 1–2 digit IDs with ~20k rows each, one (`777`) has only 635 rows and a 3-digit format. First candidate for an artificially inserted group → investigated first in notebook 02.